<a href="https://colab.research.google.com/github/lnguyen99/PyHealth/blob/master/deep_cox_mixture_all_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 598 DLH — Deep Cox Mixtures (DCM) — Colab workspace

**Team:** Jordan Alwan (jzalwan2), Jenny Nguyen (lmn4), Karthik Pullela (kpull2)  
**Paper:** Deep Cox Mixtures for Survival Regression, Nagpal et al., MLHC 2021 — https://proceedings.mlr.press/v149/nagpal21a.html  
**Deadline:** Sun Nov 29, 2026, 11:59 PM CT (9:59 PM PT)   
**Plan:** Option 2 first (DCM model on SUPPORT, 50 pts), then Option 4 if time (add FLCHAIN dataset + task, +10 EC). See the checklist Google Doc in this Drive folder.  
**PR branch:** PyHealth `master`.  
**Datasets:** SUPPORT (for Option 2), and maybe FLCHAIN (for Option 4).  
**SEER:** out of scope (DUA + no 168-feature list / cohort definition in the paper + 8-day deadline — see checklist for the paste-ready explanation).  

Run cells top to bottom. Do NOT `pip install auton-survival` — it pins torch 1.13 / numpy 1.24 / pandas 1.5 and will clash with PyHealth. Port its DCM code instead (see checklist Phase 2).

In [ ]:
# 0 — Environment check (run first)
import sys, torch
print('python', sys.version)
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

python 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
torch 2.14.1+cu130 cuda available: True
gpu: Tesla T4


In [ ]:
# 1 — Clone PyHealth off master and install
# Run this cell, then RESTART the runtime (Runtime > Restart runtime) before
# running the next cell. Editable installs often are not importable in the
# same kernel session — that is what caused ModuleNotFoundError in the first run.
# No -q here, so a failed install shows its real error instead of hiding it.
!git clone --depth 1 --branch master https://github.com/sunlabuiuc/PyHealth.git /content/PyHealth
!pip install -e /content/PyHealth scikit-survival scipy


Cloning into '/content/PyHealth'...
remote: Enumerating objects: 1861, done.
remote: Counting objects: 100% (1861/1861), done.
remote: Compressing objects: 100% (1623/1623), done.
remote: Total 1861 (delta 261), reused 1311 (delta 219), pack-reused 0 (from 0)
Receiving objects: 100% (1861/1861), 24.89 MiB | 30.42 MiB/s, done.
Resolving deltas: 100% (261/261), done.
Obtaining file:///content/PyHealth
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 16.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 12.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 64.6 MB/

In [ ]:
# 1b — Verify the install (run AFTER restarting the runtime)
import sys
sys.path.append('/content/PyHealth')

import pyhealth
print('pyhealth imported from:', pyhealth.__file__)
import sklearn, scipy
print('scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)

pyhealth imported from: /content/PyHealth/pyhealth/__init__.py
scikit-learn 1.9.1 | scipy 1.16.3


## 2 — SUPPORT smoke test (Phase 1, Option 2)

SUPPORT is public, no registration. The next cell downloads it directly from UCI (`https://archive.ics.uci.edu/static/public/880/data.csv` — the same URL PyHealth's own examples use) and renames its `id` column to `sno`, which `Support2Dataset` expects. No manual upload needed.

Reuse the existing task `SurvivalTimeToEventSupport2`: 27 features (8 numeric + 19 drop-first one-hots), complete-case 8,873 of 9,105 patients. The paper reports 44 features / 9,105 patients with no preprocessing spec — document the difference, do not claim an exact match.

In [ ]:
# 2a — Download SUPPORT (UCI, public, no registration)
from pathlib import Path
from urllib.request import urlopen
import pandas as pd

root = Path('/content/data/support2')
root.mkdir(parents=True, exist_ok=True)
csv_path = root / 'support2.csv'
if not csv_path.exists():
    url = 'https://archive.ics.uci.edu/static/public/880/data.csv'
    with urlopen(url, timeout=30) as r:
        frame = pd.read_csv(r).rename(columns={'id': 'sno'})
    assert {'sno', 'd.time', 'death'}.issubset(frame.columns), frame.columns.tolist()
    frame.to_csv(csv_path, index=False)
    print('downloaded', csv_path)
frame = pd.read_csv(csv_path)
print(frame.shape, '(expect 9105 rows)')
assert len(frame) == 9105, len(frame)
print('SUPPORT download OK')

downloaded /content/data/support2/support2.csv
(9105, 48) (expect 9105 rows)
SUPPORT download OK


In [ ]:
# 2b — SUPPORT smoke test via PyHealth
from pyhealth.datasets import Support2Dataset
from pyhealth.tasks import SurvivalTimeToEventSupport2

dataset = Support2Dataset(root='/content/data/support2', tables=['support2'])
samples = dataset.set_task(SurvivalTimeToEventSupport2())
print('n samples:', len(samples), '(expect 8873 complete-case)')
assert len(samples) == 8873, len(samples)
print('first sample keys:', samples[0].keys())
print('feature dim:', len(samples[0]['features']), '(expect 27)')
assert len(samples[0]['features']) == 27
print('duration/event:', samples[0]['duration'], samples[0]['event'])
print('SMOKE TEST PASSED: SUPPORT loads via PyHealth')

No config path provided, using default config
Initializing support2 dataset from /content/data/support2 (dev mode: False)
No cache_dir given: PyHealth will write processed copies of this dataset (event table, task samples) to /root/.cache/pyhealth/b9cb987e-c002-5476-93c0-0bd402e0208a. For identified or sensitive data, pass cache_dir=... inside your project's controlled storage, or set PYHEALTH_REQUIRE_CACHE_DIR=1 to make a missing cache_dir an error.
Setting task SurvivalTimeToEventSupport2 for support2 base dataset...
Task cache paths: task_df=/root/.cache/pyhealth/b9cb987e-c002-5476-93c0-0bd402e0208a/tasks/SurvivalTimeToEventSupport2_14f45537-6fc5-5e8f-8620-27bc96ee01f9/task_df.ld, samples=/root/.cache/pyhealth/b9cb987e-c002-5476-93c0-0bd402e0208a/tasks/SurvivalTimeToEventSupport2_14f45537-6fc5-5e8f-8620-27bc96ee01f9/samples_8d49b860-4994-5238-8a90-156b27a35d31.ld
Applying task transformations on data with 1 workers...
No cached event dataframe found. Creating: /root/.cache/pyhealth/

  0%|          | 0/9105 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


100%|██████████| 9105/9105 [00:09<00:00, 959.32it/s] 

Worker 0 finished processing patients.
Fitting processors on the dataset...


Processing samples and saving to /root/.cache/pyhealth/b9cb987e-c002-5476-93c0-0bd402e0208a/tasks/SurvivalTimeToEventSupport2_14f45537-6fc5-5e8f-8620-27bc96ee01f9/samples_8d49b860-4994-5238-8a90-156b27a35d31.ld...
Applying processors on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 8873 samples. (0 to 8873)


  0%|          | 0/8873 [00:00<?, ?it/s]

Rank 0 inferred the following `['no_header_tensor:1', 'no_header_tensor:1', 'no_header_tensor:1', 'bytes']` data format.


100%|██████████| 8873/8873 [00:00<00:00, 11243.53it/s]

Worker 0 finished processing samples.
Cached processed samples to /root/.cache/pyhealth/b9cb987e-c002-5476-93c0-0bd402e0208a/tasks/SurvivalTimeToEventSupport2_14f45537-6fc5-5e8f-8620-27bc96ee01f9/samples_8d49b860-4994-5238-8a90-156b27a35d31.ld
n samples: 8873 (expect 8873 complete-case)
first sample keys: dict_keys(['patient_id', 'features', 'duration', 'event'])
feature dim: 27 (expect 27)
duration/event: tensor([2029.]) tensor([0.])
SMOKE TEST PASSED: SUPPORT loads via PyHealth


## 3 — FLCHAIN (Part 2, Option 4 only — do this AFTER the Part 1 PR is open)

Public, no registration. Primary source: https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/survival/flchain.csv — do NOT vendor the CSV into the PyHealth repo; download at runtime. Drop `chapter` (leaks outcome), drop only the 1,350 rows missing `creatinine` → 6,524 patients. One-hot `sex` / `flc.grp` / `sample.yr` → 26 model inputs. Note: the paper says minority 'Female 44.94%' but the data are F 55.06% / M 44.94% — 44.94% is the male share; report what you compute.

In [ ]:
# 3 — FLCHAIN download + sanity check (Part 2 only)
import pandas as pd
url = 'https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/survival/flchain.csv'
df = pd.read_csv(url, index_col=0)
print(df.shape, '(raw, expect 7874 x 11)')
assert df.shape == (7874, 11), df.shape
print(df.columns.tolist())
cc = df.dropna(subset=['creatinine']).drop(columns=['chapter'])
print('complete-case:', len(cc), '(expect 6524)')
assert len(cc) == 6524, len(cc)
print('death rate:', cc['death'].mean(), 'censoring:', 1 - cc['death'].mean())
assert abs(cc['death'].mean() - 0.301) < 0.01
print('sex counts:', cc['sex'].value_counts(normalize=True).to_dict())
print('event-time quantiles (days):', cc.loc[cc['death']==1, 'futime'].quantile([0.25, 0.5, 0.75]).tolist())
print('SMOKE TEST PASSED: FLCHAIN loads')

(7874, 11) (raw, expect 7874 x 11)
['age', 'sex', 'sample.yr', 'kappa', 'lambda', 'flc.grp', 'creatinine', 'mgus', 'futime', 'death', 'chapter']
complete-case: 6524 (expect 6524)
death rate: 0.30073574494175354 censoring: 0.6992642550582464
sex counts: {'F': 0.5505824647455548, 'M': 0.4494175352544451}
event-time quantiles (days): [902.25, 2084.0, 3245.0]
SMOKE TEST PASSED: FLCHAIN loads


## Next steps (see checklist)

- Phase 2: port DCM (`DeepCoxMixturesTorch` + EM driver from auton-survival, by reading — do not install it) into `pyhealth/models/deep_cox_mixtures.py`; EM loop lives in the example script, not in `Trainer.train()` (Decision 2, recommended option b).
- Phase 3: 1-fold run, then 5-fold CV on SUPPORT; compare to paper targets (SUPPORT 75th quantile: C-td 0.6753, AUC 0.7256, ECE 0.0256, Brier 0.2118).
- Phase 4: ablations K=1 vs K>1, and hard vs *true* soft assignment (auton-survival's `typ='soft'` is sampled-hard, not soft — new code needed).
- Phase 5: synthetic-only tests, RST docs, open the PR on `master`.
- Decision Gate, then Part 2 (FLCHAIN) only if the PR is open, CI is green, and it is Nov 25 or earlier.

## Smoke-test result

If every cell above ran without an `AssertionError`, both datasets load:

- **SUPPORT:** 9,105 rows downloaded → 8,873 complete-case samples via `Support2Dataset` + `SurvivalTimeToEventSupport2`, 27 features.
- **FLCHAIN:** 7,874 raw rows → 6,524 complete-case rows, death rate ≈ 0.301.

Next: Phase 2 in the checklist — port DCM into `pyhealth/models/deep_cox_mixtures.py`.